# SPY 0DTE Long Iron Condor — Colab Runner

Run the backtest from a phone or any browser. You need a Polygon Options Advanced/Developer key.

**Strategy in one line:** every minute from 9:50 ET, look at RSI on 1-min SPY; the first time it crosses above 70 or below 30 inside the entry window, buy a long iron condor and exit on the first of (profit target, stop loss, time stop).

Cells below: install → key → smoke test → short sweep → full year → top results.

## 1. Clone the repo and install
Re-running this cell is a no-op once the repo is cloned.

In [ ]:
import os, sys, subprocess
REPO_DIR = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO_DIR}

%cd {REPO_DIR}
!git pull --quiet
!pip install -q -r requirements.txt -e .
print('OK')

## 2. Paste your Polygon API key
Uses `getpass` so the key is not saved in the notebook. If the cell is re-run, paste again.

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent trading day
Confirms SPY bars + option chain + 4-leg pricing + signal logic all wire up. Takes ~30 s.

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Optional — persist the Polygon cache to Google Drive
Colab kills idle sessions after ~90 min and the local disk is wiped. Mount Drive so option bars are cached across sessions; second runs will be much faster.

Skip this cell if you do not care about persistence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
PERSIST = '/content/drive/MyDrive/iron-condor-cache'
LOCAL   = '/content/iron-condor/data/cache'
os.makedirs(PERSIST, exist_ok=True)
if os.path.islink(LOCAL):
    pass
else:
    if os.path.isdir(LOCAL):
        shutil.rmtree(LOCAL)
    os.symlink(PERSIST, LOCAL)
print('cache now lives at', PERSIST)

## 5. Short sweep — last 30 trading days
Full grid (2 RSI × 7 strike rules × 5 PT × 3 SL × 4 cutoffs = 840 configs) on 30 days. Useful sanity check before paying for a year of fetches. Expect a few minutes.

In [ ]:
!python -m iron_condor.cli --days 30 --sweep

## 6. Full year sweep
Real run. Most of the time is Polygon fetches the first pass; subsequent runs hit the cache. Plan ~15–30 min.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 7. Look at the results
Top 20 configs by total return, plus a slice by profit-target and stop-loss buckets.

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

In [ ]:
# Pivot: average return by profit-target × stop-loss across all other params
trades = pd.read_csv('results/sweep_trades.csv')
pivot = (
    summary.assign(
        pt=summary['config'].str.extract(r'pt(\d+)')[0].astype(int),
        sl=summary['config'].str.extract(r'sl(\d+)')[0].astype(int),
    )
    .pivot_table(index='pt', columns='sl', values='total_return_pct', aggfunc='mean')
)
pivot.round(1)

In [ ]:
# Distribution of exit reasons across all trades
trades['exit_reason'].value_counts()